[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/moienr/AL_Training/blob/main/notebooks/03_global_map.ipynb)

# Part 3: the global campaign: its method, the 54,942 positives it found, and the Austrian inventory

The EarthQuery campaign ran the loop of part 2 in more than 100 regions around the world. Instead of a person, a jury of two vision language models labelled the candidates on satellite basemaps:

![the VLM campaign](https://raw.githubusercontent.com/moienr/AL_Training/main/figures/vlm_campaign.png)

Each region starts from the verified solar farms of the Microsoft global layer, borrowing the most similar sites from other regions when it has fewer than 100, plus 200 representative landscape negatives. It is then searched in seven passes: three at 2,560 m, two at 160 m, two at 20 m, the model retrained after every pass. Every candidate gets four votes, Gemma 4 and Qwen 3.6 each on a Google and an Esri image. Two yes votes on one basemap make a positive, four no votes a negative, and anything else is a tie for a person. This notebook shows the positives it produced, by wave.

In [ ]:
# On Google Colab this cell fetches the course files and installs what Colab lacks; elsewhere it does nothing.
import os, sys, subprocess
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    if not os.path.isdir("/content/AL_Training"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/moienr/AL_Training.git", "/content/AL_Training"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + ['plotly>=5.24'], check=True)
    os.chdir("/content/AL_Training/notebooks")

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath(".."))      # makes `al_training` importable from this folder
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
os.makedirs("../outputs", exist_ok=True)

In [ ]:
from al_training.globalmap import load_points

P, S = load_points()
print(f"{len(P):,} campaign positives ({int(P.hand_labelled.sum()):,} labelled by people), {len(S):,} Microsoft sites")
P.head()

## The campaign positives and the Microsoft seed sites

Teal: the campaign positives. Gold, drawn underneath: the 62,172 verified solar farms of the Microsoft global layer, which seeded the models.

In [ ]:
from al_training.globalmap import draw_world, TEAL, GOLD

fig, ax = plt.subplots(figsize=(11, 5.4))
draw_world(ax)
ax.scatter(S["lng"], S["lat"], s=1.2, color=GOLD, lw=0, alpha=0.8, label=f"Microsoft sites ({len(S):,})")
ax.scatter(P["lng"], P["lat"], s=1.2, color=TEAL, lw=0, alpha=0.9, label=f"campaign positives ({len(P):,})")
ax.legend(loc="lower left", markerscale=8, frameon=False);

## Animation: the positives added at each stage, pilot regions, waves 1 to 6 and the extra passes

The points appear in the order the campaign produced them: the three pilot regions labelled by people, then waves 1 to 6 in the order the regions finished (28 August to 4 September 2026), then the extra passes over finished regions.

In [ ]:
from IPython.display import HTML
from al_training.globalmap import WorldFrames
from al_training.plotting import save_gif

world = WorldFrames(P, S, n_frames=40)
anim = world.animation(interval=150)
save_gif(anim, "../outputs/global_map.gif", fps=6)
HTML(anim.to_jshtml())

## The same stages as an interactive map: slider, zoom, and the region of each point on hover

Drag the slider or press play. Hover over a point to see its region. The map can be zoomed.

In [ ]:
from al_training.globalmap import plotly_animation

plotly_animation(P, S).show()

## Hit rates by pass: coarse passes find negatives, fine passes find farms

The coarse passes mostly find negatives: 86 to 90% of the candidates at 2,560 m are not farms. The fine passes find farms: 33 to 53% of the candidates at 20 m are positive, and the hit rate rises from one pass to the next (in China 9 from 15% to 77%). Across waves 1 to 4, which cover 101 regions, the jury looked at 80,908 candidates and returned 25,961 positives, 47,618 negatives and 7,329 ties, from 323,922 votes. Exploit has the highest yield in every region; novelty mostly collects hard negatives in sparse regions and finds farms in dense ones.

## The Austrian inventory: 355 solar farms with outlines

Where a person ran the loop to the end, the result is a complete inventory. Austria: 355 solar farms covering 933 ha of panels (median 1.1 ha, the largest 90 ha), each outlined by SAM 3 on 0.3 m imagery. Before the Microsoft points were used, the map held 234 sites and the Microsoft layer 136, with 102 within 500 m of one another; after their points joined the training set the final map holds 355, that is 121 more than ours had: the 34 that only Microsoft held and 87 that the re-run found beyond both.

![the Austrian product](https://raw.githubusercontent.com/moienr/AL_Training/main/figures/austria_product.png)

## Summary

* One loop, one seed layer, one jury: the campaign produced about 55,000 positives in about a week of compute.
* The map follows the Microsoft layer where that layer is dense and adds regions it does not cover.
* Run to the end in one country, the same loop gives a complete inventory with outlines.

Part 4 runs the loop live on the map, with you as the labeller.